# Stage A Calibration Diagnostics: 2-Knob Sweep

**Branch:** `paper/results`
**Solver:** legacy `relax_to_steady`, no distributed leakage, no BE-CV
**Tests:** 0116, 0121 (17 ft) | 0123, 0126 (15 ft)
**Calibration knobs:** A_leak_cm2, K_minor_mult (Cd and K_stove fixed)

Parts:
1. Baseline diagnostics (T, P overlays + residual profiles)
2. 2-knob Latin Hypercube sweep + local refinement
3. Side-by-side comparison (baseline vs calibrated)
4. Physics sanity checks + sensitivity

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import time as _time
import datetime
import json
import hashlib
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d
from scipy.optimize import minimize

REPO_ROOT = Path('..')
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'dev' / 'exp_data'))

from flued_module.chimney import Chimney
from flued_module.material_properties import MaterialProperties
from experiment_loader import ExperimentLoader

matplotlib.rcParams['figure.dpi'] = 120

mat_props = MaterialProperties(str(REPO_ROOT / 'flued_module' / 'data' / 'pipe_props.json'))
el = ExperimentLoader(
    metadata_path=REPO_ROOT / 'dev' / 'exp_data' / 'experiments_metadata.json'
)
print('Setup complete')

## Configuration

In [ ]:
# --- Tests ---
TEST_IDS = ['0116', '0121', '0123', '0126']

# --- Fixed parameters (NOT calibrated) ---
P_ATM          = 98457
TC1_OFFSET_C   = 10.0
DOOR_WIDTH_IN  = 15.25
DOOR_HEIGHT_IN =  9.875
DOOR_RAMP_S    =  5.0
DT             =  1.0
SIM_TIME_MAX   = 1200

# --- Fixed Stage A knobs ---
CD_FIXED       = 0.62
K_STOVE_FIXED  = 0.0

# --- Baseline values for the 2 free knobs ---
BASELINE_A_LEAK_CM2   = 75.0
BASELINE_K_MINOR_MULT = 1.0

# --- Geometry ---
DIAMETER_IN  = 6.0
DIAMETER_FT  = DIAMETER_IN / 12
DIAMETER_M   = DIAMETER_IN * 0.0254
PIPE_AREA_M2 = np.pi * (DIAMETER_M / 2)**2

FINE_TOP_IN   = 15
FINE_TOP_SEGS = 15
BLACK_PIPE_IN  = 73.5
TEE1_VERT_IN   = 12.0;  TEE1_HORIZ_IN  = 6.5
ELBOW_VERT_IN  =  7.5;  ELBOW_HORIZ_IN = 8.5
HORIZ_IN       = 23.25
TEE2_VERT_IN   = 12.0;  TEE2_HORIZ_IN  = 6.5

PRESSURE_SUBSAMPLE = 10

# --- Sensor maps ---
SENSOR_MAP = {
    '15ft': {'TC1': (0,15.0), 'TC2': (2,12.0), 'TC3': (2,42.0), 'TOP': (3,3.0)},
    '17ft': {'TC1': (0,15.0), 'TC6': (2,3.0), 'TC2': (2,34.75), 'TC3': (2,64.75), 'TOP': (3,3.0)},
}
TEMP_SENSORS  = {'15ft': ['TC1','TC2','TC3'], '17ft': ['TC1','TC6','TC2','TC3']}
PRESS_SENSORS = {'15ft': ['TC1','TC2','TC3','TOP'], '17ft': ['TC1','TC6','TC2','TC3','TOP']}
SENSOR_COLORS = {'TC1':'C0', 'TC6':'C1', 'TC2':'C2', 'TC3':'C3', 'TOP':'C4'}

# --- Calibration sweep config ---
N_COARSE       = 10     # LHS samples (set 50 for production, 10 for smoke test)
N_REFINE_TOP   = 3      # top candidates to refine
REFINE_MAXITER = 15     # Nelder-Mead max iterations per candidate
RNG_SEED       = 42

# --- Sweep bounds (2 knobs only) ---
A_LEAK_BOUNDS      = (40.0, 110.0)   # cm^2
K_MINOR_MULT_BOUNDS = (0.7, 2.0)

# --- Objective weights ---
W_MDOT  = 1.0
W_PRESS = 1.0
SIGMA_P = 1.0     # Pa
EXCLUDE_PRESS_SENSORS = ['TC1']  # TC1 is DG, exclude from pressure objective

# --- Output ---
TIMESTAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_ROOT  = REPO_ROOT / 'outputs' / 'calibration_stageA_diag' / TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Output -> {OUT_ROOT}')
print(f'Sweep: {N_COARSE} LHS samples, refine top {N_REFINE_TOP}')
print(f'Bounds: A_leak={A_LEAK_BOUNDS}, K_minor_mult={K_MINOR_MULT_BOUNDS}')
print(f'Fixed: Cd={CD_FIXED}, K_stove={K_STOVE_FIXED}')

## Helpers

In [ ]:
def smooth_transition(t, t_start, t_end, val_start, val_end):
    if t <= t_start: return val_start
    if t >= t_end:   return val_end
    frac = (t - t_start) / (t_end - t_start)
    return val_start + (val_end - val_start) * 0.5 * (1.0 - np.cos(np.pi * frac))


def build_parts(chimney_type, T_indoor_K, T_outdoor_K):
    geom = el.get_chimney_geometry(chimney_type)
    bulk_vert_in = geom['final_vert_in'] - FINE_TOP_IN
    return [
        mat_props.create_single_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=BLACK_PIPE_IN/12,
            z_ft=0, material='steel_24_gauge', orientation='vertical',
            T_ambient_K=T_indoor_K, Pa=P_ATM, num_segments=6),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=HORIZ_IN/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='horizontal',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=2),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=bulk_vert_in/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='vertical',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=8),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=FINE_TOP_IN/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='vertical',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=FINE_TOP_SEGS),
    ]


def build_fittings(T_indoor_K, T_outdoor_K, vane_factor=1.0):
    return [
        {'type': 'tee', 'between_parts': (-1, 0),
         'horizontal_length_ft': TEE1_HORIZ_IN/12,
         'vertical_length_ft':   TEE1_VERT_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'single',
         'material': 'steel_24_gauge', 'T_ambient_K': T_indoor_K,
         'vane_factor': vane_factor},
        {'type': 'elbow', 'between_parts': (0, 1),
         'height_ft': ELBOW_VERT_IN/12, 'width_ft': ELBOW_HORIZ_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'single',
         'material': 'steel_24_gauge', 'T_ambient_K': T_indoor_K,
         'angle_deg': 90, 'R_over_D': 1.5, 'elbow_type': 'long_radius',
         'vane_factor': vane_factor},
        {'type': 'tee', 'between_parts': (1, 2),
         'horizontal_length_ft': TEE2_HORIZ_IN/12,
         'vertical_length_ft':   TEE2_VERT_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'triple',
         'inner_material': 'stainless_430', 'middle_material': 'ceramic_fiber',
         'outer_material': 'stainless_316', 'T_ambient_K': T_outdoor_K,
         'vane_factor': vane_factor},
    ]


def make_door_func(test_id):
    cfg     = el.get_test_config(test_id)
    offsets = cfg['door_events']['offsets_s']
    door_in = cfg['door_events'].get('door_opening_in', 2.0)
    RAMP    = DOOR_RAMP_S
    cracked_pct = (
        np.degrees(np.arcsin(
            min(door_in, DOOR_HEIGHT_IN * 0.999) / DOOR_HEIGHT_IN
        )) / 90.0 * 100.0
    )
    door_key = 'door_to_2in' if 'door_to_2in' in offsets else 'door_to_3in'
    t_crack  = offsets[door_key]
    t_close  = offsets['door_closed']
    t_reopen = offsets.get('door_reopen_2in', None)
    t_close2 = offsets.get('door_closed_again', None)

    def door_func(t):
        if t < t_crack - RAMP:
            return 100.0
        if t < t_crack:
            return smooth_transition(t, t_crack - RAMP, t_crack, 100.0, cracked_pct)
        if t < t_close:
            return cracked_pct
        if t < t_close + RAMP:
            return smooth_transition(t, t_close, t_close + RAMP, cracked_pct, 0.0)
        if t_reopen is not None:
            if t < t_reopen:
                return 0.0
            if t < t_reopen + RAMP:
                return smooth_transition(t, t_reopen, t_reopen + RAMP, 0.0, cracked_pct)
            if t_close2 is not None and t < t_close2:
                return cracked_pct
            if t_close2 is not None and t < t_close2 + RAMP:
                return smooth_transition(t, t_close2, t_close2 + RAMP, cracked_pct, 0.0)
        return 0.0
    return door_func


def run_sim(test_id, A_leak_cm2=BASELINE_A_LEAK_CM2, K_minor_mult=BASELINE_K_MINOR_MULT):
    """Run one simulation with the 2 free knobs. Cd and K_stove are fixed."""
    cfg      = el.get_test_config(test_id)
    ct       = cfg['chimney_type']
    cond     = cfg['conditions']
    offsets  = cfg['door_events']['offsets_s']
    T_in_K   = cond['T_indoor_C']  + 273.15
    T_out_K  = cond['T_outdoor_C'] + 273.15
    wind_ms  = cond['wind_speed_m_s']

    parts    = build_parts(ct, T_in_K, T_out_K)
    fittings = build_fittings(T_in_K, T_out_K, vane_factor=K_minor_mult)
    door_fn  = make_door_func(test_id)

    tc1_data = el.get_temperature_data(test_id, 'TC1')
    tc1_K    = np.asarray(tc1_data['temperature_C'], dtype=float) + TC1_OFFSET_C + 273.15
    inlet_fn = interp1d(tc1_data['time_s'], tc1_K,
                        kind='linear', bounds_error=False, fill_value='extrapolate')

    sim_time = min(offsets.get('test_stop', SIM_TIME_MAX), SIM_TIME_MAX)

    inlet_cfg = {
        'enable': True, 'door_type': 'swivel',
        'door_width_ft':   DOOR_WIDTH_IN / 12,
        'door_height_ft':  DOOR_HEIGHT_IN / 12,
        'Cd':              CD_FIXED,
        'A_leak_m2':       A_leak_cm2 * 1e-4,
        'inlet_area_min_m2': 0.0,
        'K_stove':         K_STOVE_FIXED,
    }

    top_bnd = {
        'enable_wind_cap_effect': True,
        'cap_type':               'rain_cap',
        'wind_speed_m_s':         wind_ms,
    }

    ch = Chimney(
        parts=parts, elbows=fittings,
        dt=DT, total_time=sim_time,
        use_coupled_solver=True,
        door_opening_func=door_fn,
        inlet_temp_func=inlet_fn,
        inlet_restriction=inlet_cfg,
        top_boundary=top_bnd,
        leakage_config=None,
        enable_diagnostics=False,
        verbose=False,
    )
    ch.run_simulation()
    return ch


def find_segment(ch, part_idx, height_in):
    target_m = ch.part_base_elevations[part_idx] + height_in * 0.0254
    return int(np.argmin(np.abs(ch.segment_elevations - target_m)))


def get_phase_windows(test_id):
    offsets  = el.get_test_config(test_id)['door_events']['offsets_s']
    door_key = 'door_to_2in' if 'door_to_2in' in offsets else 'door_to_3in'
    t_crack  = offsets[door_key]
    t_close  = offsets['door_closed']
    t_end    = min(offsets.get('test_stop', SIM_TIME_MAX), SIM_TIME_MAX)
    return (t_crack, t_close, t_close + 30, t_end)


def get_door_event_times(test_id):
    offsets = el.get_test_config(test_id)['door_events']['offsets_s']
    keys = ['door_to_2in', 'door_to_3in', 'door_closed',
            'door_reopen_2in', 'door_closed_again']
    return sorted(offsets[k] for k in keys if k in offsets)


def compute_rmse_bias(exp_t, exp_v, sim_t, sim_v, t_window):
    exp_t = np.asarray(exp_t, dtype=float)
    exp_v = np.asarray(exp_v, dtype=float)
    mask  = (exp_t >= t_window[0]) & (exp_t <= t_window[1])
    et, ev = exp_t[mask], exp_v[mask]
    if len(et) < 5:
        return np.nan, np.nan
    sv   = np.interp(et, sim_t, sim_v)
    diff = sv - ev
    return np.sqrt(np.nanmean(diff**2)), float(np.nanmean(diff))


def savefig(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  saved: {path}')
    plt.close(fig)


def extract_pressure(ch, test_id):
    """Extract gauge pressure time series at sensor locations."""
    ct = exp_data[test_id]['chimney_type']
    n  = ch.T_gas.shape[0]
    seg_map = {s: find_segment(ch, *SENSOR_MAP[ct][s]) for s in PRESS_SENSORS[ct]}
    step_indices = np.arange(0, n, PRESSURE_SUBSAMPLE)
    times = step_indices * ch.dt
    unique_segs = list(set(seg_map.values()))
    seg_P = {seg: np.zeros(len(step_indices)) for seg in unique_segs}
    for i, step in enumerate(step_indices):
        prof = ch.get_static_pressure_profile(timestep=int(step))
        for seg in unique_segs:
            seg_P[seg][i] = prof['P_gauge_Pa'][seg]
    return {s: (times, seg_P[seg_map[s]]) for s in PRESS_SENSORS[ct]}


def extract_ss(ch, test_id):
    """Extract steady-state mdot, pressure, and temperature at sensors."""
    ct = exp_data[test_id]['chimney_type']
    _, _, t_cls0, t_cls1 = get_phase_windows(test_id)
    t_sim = np.arange(ch.T_gas.shape[0]) * ch.dt

    # SS mdot
    mask_ss = (t_sim >= t_cls0) & (t_sim <= t_cls1)
    mdot = ch.mass_flow_rate
    if mdot.ndim > 1:
        mdot = mdot[:, 0]
    mdot_ss = float(np.mean(mdot[mask_ss])) * 1000 if mask_ss.sum() > 0 else np.nan

    # SS pressure per sensor
    press = extract_pressure(ch, test_id)
    press_ss = {}
    for s in PRESS_SENSORS[ct]:
        pt, pv = press[s]
        pmask = (pt >= t_cls0) & (pt <= t_cls1)
        press_ss[s] = float(np.mean(pv[pmask])) if pmask.sum() > 0 else np.nan

    # SS temperature per sensor
    temp_ss = {}
    for s in TEMP_SENSORS[ct]:
        seg = find_segment(ch, *SENSOR_MAP[ct][s])
        T_sim = ch.T_gas[:, seg] - 273.15
        tmask = (t_sim >= t_cls0) & (t_sim <= t_cls1)
        temp_ss[s] = float(np.mean(T_sim[tmask])) if tmask.sum() > 0 else np.nan

    # Temperature RMSE per sensor
    temp_rmse = {}
    for s in TEMP_SENSORS[ct]:
        seg = find_segment(ch, *SENSOR_MAP[ct][s])
        T_sim = ch.T_gas[:, seg] - 273.15
        td = exp_data[test_id]['temperature'][s]
        et = np.asarray(td['time_s'], dtype=float)
        ev = np.asarray(td['temperature_C'], dtype=float)
        rmse, bias = compute_rmse_bias(et, ev, t_sim, T_sim, (t_cls0, t_cls1))
        temp_rmse[s] = {'rmse': rmse, 'bias': bias}

    return {
        'mdot_ss': mdot_ss,
        'press_ss': press_ss,
        'temp_ss': temp_ss,
        'temp_rmse': temp_rmse,
    }


def get_pitot_mdot(test_id):
    dp_data  = el.get_sensor_data(test_id, 'pitot_dp', as_gauge=False)
    tc4_data = el.get_temperature_data(test_id, 'TC4')
    top_data = el.get_sensor_data(test_id, 'TOP', as_gauge=True)
    t_dp = np.asarray(dp_data['time_s'],      dtype=float)
    dp   = np.asarray(dp_data['pressure_Pa'],  dtype=float)
    t_tc4 = np.asarray(tc4_data['time_s'],        dtype=float)
    T_tc4 = np.asarray(tc4_data['temperature_C'], dtype=float) + 273.15
    t_top = np.asarray(top_data['time_s'],      dtype=float)
    P_top = np.asarray(top_data['pressure_Pa'], dtype=float)
    T_g = np.interp(t_dp, t_tc4, T_tc4)
    P_g = np.interp(t_dp, t_top, P_top)
    R_AIR = 287.0
    rho   = (P_ATM + P_g) / (R_AIR * np.maximum(T_g, 200.0))
    v     = np.sqrt(2.0 * np.maximum(dp, 0.0) / np.maximum(rho, 0.1))
    mdot  = rho * v * PIPE_AREA_M2
    dt_est = max(float(np.median(np.diff(t_dp))), 0.1)
    win    = max(1, int(10.0 / dt_est))
    mdot_sm = uniform_filter1d(mdot, size=win)
    return t_dp, mdot * 1e3, mdot_sm * 1e3


def get_sensor_elevation_m(ch, test_id, sensor):
    """Absolute elevation of a sensor in metres."""
    ct = exp_data[test_id]['chimney_type']
    part_idx, h_in = SENSOR_MAP[ct][sensor]
    return ch.part_base_elevations[part_idx] + h_in * 0.0254


print('Helpers defined.')

## Load experimental data

In [ ]:
exp_data = {}
for test_id in TEST_IDS:
    cfg = el.get_test_config(test_id)
    ct  = cfg['chimney_type']
    temp  = {s: el.get_temperature_data(test_id, s) for s in TEMP_SENSORS[ct]}
    press = {s: el.get_sensor_data(test_id, s, as_gauge=True) for s in PRESS_SENSORS[ct]}
    exp_data[test_id] = {
        'chimney_type': ct, 'config': cfg,
        'temperature': temp, 'pressure': press,
    }
    cond = cfg['conditions']
    print(f'{test_id} ({ct}): wind={cond["wind_speed_m_s"]:.1f} m/s, '
          f'T_in={cond["T_indoor_C"]:.0f} C, T_out={cond["T_outdoor_C"]:.0f} C')

# Pitot-derived experimental SS mdot
_pj = REPO_ROOT / 'dev' / 'notebooks' / 'validation' / 'pitot_calibration_results.json'
with open(_pj) as f:
    pitot_cal = json.load(f)

exp_ss_mdot = {}
exp_ss_mdot_std = {}
for tid in TEST_IDS:
    if tid in pitot_cal:
        exp_ss_mdot[tid]     = pitot_cal[tid]['experimental']['mdot_kgs'] * 1000
        exp_ss_mdot_std[tid] = pitot_cal[tid]['experimental']['mdot_std_kgs'] * 1000
    else:
        exp_ss_mdot[tid]     = np.nan
        exp_ss_mdot_std[tid] = np.nan

# Experimental SS pressure at each sensor
exp_ss_press = {}
exp_ss_temp  = {}
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    _, _, t_cls0, t_cls1 = get_phase_windows(tid)
    exp_ss_press[tid] = {}
    for s in PRESS_SENSORS[ct]:
        pd_ = exp_data[tid]['pressure'][s]
        et = np.asarray(pd_['time_s'], dtype=float)
        ev = np.asarray(pd_['pressure_Pa'], dtype=float)
        mask = (et >= t_cls0) & (et <= t_cls1)
        exp_ss_press[tid][s] = float(np.mean(ev[mask])) if mask.sum() > 0 else np.nan
    exp_ss_temp[tid] = {}
    for s in TEMP_SENSORS[ct]:
        td = exp_data[tid]['temperature'][s]
        et = np.asarray(td['time_s'], dtype=float)
        ev = np.asarray(td['temperature_C'], dtype=float)
        mask = (et >= t_cls0) & (et <= t_cls1)
        exp_ss_temp[tid][s] = float(np.mean(ev[mask])) if mask.sum() > 0 else np.nan

print(f'Loaded {len(exp_data)} tests.')
print('Exp SS mdot (g/s):', {k: f"{v:.1f}" for k, v in exp_ss_mdot.items()})

---
## Part 1: Baseline Diagnostics

Run baseline parameters (A_leak=75, K_minor_mult=1.0, Cd=0.62, K_stove=0.0).
Full temperature + pressure overlays, steady-state profiles, and residual plots.

In [ ]:
# Run baseline simulations
print('Running baseline simulations...')
baseline = {}
t0 = _time.time()
for tid in TEST_IDS:
    ch = run_sim(tid, A_leak_cm2=BASELINE_A_LEAK_CM2, K_minor_mult=BASELINE_K_MINOR_MULT)
    ss = extract_ss(ch, tid)
    press = extract_pressure(ch, tid)
    baseline[tid] = {'ch': ch, 'ss': ss, 'press': press}
    print(f'  {tid}: mdot_ss={ss["mdot_ss"]:.1f} g/s  ({_time.time()-t0:.0f}s)')
print(f'Baseline done in {_time.time()-t0:.0f}s')

### 1a. Temperature overlays (full time)

In [ ]:
baseline_dir = OUT_ROOT / 'baseline'
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ch = baseline[tid]['ch']
    t_sim = np.arange(ch.T_gas.shape[0]) * ch.dt
    door_times = get_door_event_times(tid)
    sensors = TEMP_SENSORS[ct]

    fig, axes = plt.subplots(len(sensors), 1, figsize=(12, 3*len(sensors)), sharex=True)
    if len(sensors) == 1: axes = [axes]

    for ax, sensor in zip(axes, sensors):
        td = exp_data[tid]['temperature'][sensor]
        et = np.asarray(td['time_s'], dtype=float)
        ev = np.asarray(td['temperature_C'], dtype=float)

        seg = find_segment(ch, *SENSOR_MAP[ct][sensor])
        sim_T = ch.T_gas[:, seg] - 273.15

        ax.plot(et/60, ev, 'k-', lw=0.8, alpha=0.7, label=f'{sensor} exp')
        ax.plot(t_sim/60, sim_T, color=SENSOR_COLORS[sensor], ls='--', lw=1.3,
                label=f'{sensor} sim')
        for dt_ in door_times:
            ax.axvline(dt_/60, color='gray', ls=':', alpha=0.4)
        ax.set_ylabel(f'{sensor} T (C)')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (min)')
    axes[0].set_title(f'Baseline Temperature: {tid} ({ct})')
    savefig(fig, baseline_dir / f'{tid}_T_overlay.png')
    plt.show()

### 1b. Pressure overlays (full time)

In [ ]:
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ch = baseline[tid]['ch']
    press = baseline[tid]['press']
    door_times = get_door_event_times(tid)
    sensors = PRESS_SENSORS[ct]

    fig, axes = plt.subplots(len(sensors), 1, figsize=(12, 2.8*len(sensors)), sharex=True)
    if len(sensors) == 1: axes = [axes]

    for ax, sensor in zip(axes, sensors):
        pd_ = exp_data[tid]['pressure'][sensor]
        et = np.asarray(pd_['time_s'], dtype=float)
        ev = np.asarray(pd_['pressure_Pa'], dtype=float)
        pt, pv = press[sensor]

        ax.plot(et/60, ev, 'k-', lw=0.8, alpha=0.7, label=f'{sensor} exp')
        ax.plot(pt/60, pv, color=SENSOR_COLORS[sensor], ls='--', lw=1.3,
                label=f'{sensor} sim')
        ax.axhline(0, color='gray', ls=':', alpha=0.4)
        for dt_ in door_times:
            ax.axvline(dt_/60, color='gray', ls=':', alpha=0.4)
        ax.set_ylabel(f'{sensor} P (Pa)')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (min)')
    axes[0].set_title(f'Baseline Pressure: {tid} ({ct})')
    savefig(fig, baseline_dir / f'{tid}_P_overlay.png')
    plt.show()

### 1c. Steady-state summary

In [ ]:
# SS mdot bar chart
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(TEST_IDS))
w = 0.35
sim_mdots = [baseline[t]['ss']['mdot_ss'] for t in TEST_IDS]
exp_mdots = [exp_ss_mdot[t] for t in TEST_IDS]
exp_stds  = [exp_ss_mdot_std[t] for t in TEST_IDS]
ax.bar(x - w/2, sim_mdots, w, label='Sim (baseline)', color='C0', alpha=0.85)
ax.bar(x + w/2, exp_mdots, w, yerr=exp_stds, label='Exp (pitot)',
       color='C1', alpha=0.85, capsize=4)
for i, tid in enumerate(TEST_IDS):
    ratio = sim_mdots[i] / exp_mdots[i] if exp_mdots[i] > 0 else np.nan
    ytop = max(sim_mdots[i], exp_mdots[i] + exp_stds[i]) + 0.5
    ax.text(i, ytop, f'{ratio:.2f}x', ha='center', fontsize=9, color='dimgray')
ax.set_xticks(x)
xlabels = [tid + ' (' + exp_data[tid]['chimney_type'] + ')' for tid in TEST_IDS]
ax.set_xticklabels(xlabels)
ax.set_ylabel('SS mdot (g/s)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_title('Baseline: SS mass flow comparison')
savefig(fig, baseline_dir / 'fig_mdot_ss.png')
plt.show()

In [ ]:
# SS height profiles: pressure and temperature
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ch = baseline[tid]['ch']
    ss = baseline[tid]['ss']

    # Collect sensor elevations, values
    sensors_p = PRESS_SENSORS[ct]
    sensors_t = TEMP_SENSORS[ct]

    heights_p_sim = [get_sensor_elevation_m(ch, tid, s) for s in sensors_p]
    vals_p_sim    = [ss['press_ss'][s] for s in sensors_p]
    vals_p_exp    = [exp_ss_press[tid].get(s, np.nan) for s in sensors_p]

    heights_t_sim = [get_sensor_elevation_m(ch, tid, s) for s in sensors_t]
    vals_t_sim    = [ss['temp_ss'][s] for s in sensors_t]
    vals_t_exp    = [exp_ss_temp[tid].get(s, np.nan) for s in sensors_t]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Pressure profile
    ax1.plot(vals_p_exp, heights_p_sim, 'ko-', lw=1.2, ms=6, label='Exp')
    ax1.plot(vals_p_sim, heights_p_sim, 's--', color='C0', lw=1.2, ms=6, label='Sim')
    for s, h, ve, vs in zip(sensors_p, heights_p_sim, vals_p_exp, vals_p_sim):
        ax1.annotate(s, (vs, h), textcoords='offset points', xytext=(5,0), fontsize=8)
    ax1.set_xlabel('Gauge Pressure (Pa)')
    ax1.set_ylabel('Height (m)')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f'{tid}: SS Pressure Profile')

    # Temperature profile
    ax2.plot(vals_t_exp, heights_t_sim, 'ko-', lw=1.2, ms=6, label='Exp')
    ax2.plot(vals_t_sim, heights_t_sim, 's--', color='C0', lw=1.2, ms=6, label='Sim')
    for s, h, ve, vs in zip(sensors_t, heights_t_sim, vals_t_exp, vals_t_sim):
        ax2.annotate(s, (vs, h), textcoords='offset points', xytext=(5,0), fontsize=8)
    ax2.set_xlabel('Temperature (C)')
    ax2.set_ylabel('Height (m)')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_title(f'{tid}: SS Temperature Profile')

    savefig(fig, baseline_dir / f'{tid}_ss_profiles.png')
    plt.show()

### 1d. Residual vs height (steady-state)

In [ ]:
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ch = baseline[tid]['ch']
    ss = baseline[tid]['ss']

    sensors_p = PRESS_SENSORS[ct]
    sensors_t = TEMP_SENSORS[ct]

    heights_p = [get_sensor_elevation_m(ch, tid, s) for s in sensors_p]
    resid_p   = [ss['press_ss'][s] - exp_ss_press[tid].get(s, np.nan) for s in sensors_p]

    heights_t = [get_sensor_elevation_m(ch, tid, s) for s in sensors_t]
    resid_t   = [ss['temp_ss'][s] - exp_ss_temp[tid].get(s, np.nan) for s in sensors_t]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.barh(range(len(sensors_p)), resid_p, color='C0', alpha=0.8)
    ax1.set_yticks(range(len(sensors_p)))
    ax1.set_yticklabels([f'{s} ({h:.2f}m)' for s, h in zip(sensors_p, heights_p)])
    ax1.axvline(0, color='k', lw=0.8)
    ax1.set_xlabel('P_sim - P_exp (Pa)')
    ax1.set_title(f'{tid}: Pressure Residual')
    ax1.grid(axis='x', alpha=0.3)
    for i, v in enumerate(resid_p):
        if not np.isnan(v):
            ax1.text(v, i, f' {v:+.2f}', va='center', fontsize=8)

    ax2.barh(range(len(sensors_t)), resid_t, color='C3', alpha=0.8)
    ax2.set_yticks(range(len(sensors_t)))
    ax2.set_yticklabels([f'{s} ({h:.2f}m)' for s, h in zip(sensors_t, heights_t)])
    ax2.axvline(0, color='k', lw=0.8)
    ax2.set_xlabel('T_sim - T_exp (C)')
    ax2.set_title(f'{tid}: Temperature Residual')
    ax2.grid(axis='x', alpha=0.3)
    for i, v in enumerate(resid_t):
        if not np.isnan(v):
            ax2.text(v, i, f' {v:+.1f}', va='center', fontsize=8)

    savefig(fig, baseline_dir / f'{tid}_residuals.png')
    plt.show()

# Summary table
print('')
print('Baseline RMSE summary (closed-door):')
rows = []
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ss = baseline[tid]['ss']
    for s in TEMP_SENSORS[ct]:
        r = ss['temp_rmse'][s]
        rows.append({'Test': tid, 'Sensor': s, 'RMSE_C': r['rmse'], 'Bias_C': r['bias']})
df_base = pd.DataFrame(rows)
print(df_base.to_string(index=False, float_format='%.2f'))

---
## Part 2: Restricted Stage A Calibration (2 knobs)

**Free:** A_leak_cm2, K_minor_mult
**Fixed:** Cd=0.62, K_stove=0.0

Latin Hypercube Sampling + Nelder-Mead local refinement.

In [ ]:
# --- LHS sampling ---
def latin_hypercube_2d(n_samples, bounds, rng):
    """LHS for 2 parameters. Returns (n_samples, 2) array."""
    result = np.zeros((n_samples, 2))
    for d in range(2):
        perm = rng.permutation(n_samples)
        lo, hi = bounds[d]
        for i in range(n_samples):
            lo_bin = perm[i] / n_samples
            hi_bin = (perm[i] + 1) / n_samples
            u = rng.uniform(lo_bin, hi_bin)
            result[i, d] = lo + u * (hi - lo)
    return result


def compute_objective(A_leak_cm2, K_minor_mult):
    """Run all tests, compute J_A. Returns (J_A, details_dict)."""
    ss_mdot_sim = {}
    ss_press_sim = {}
    for tid in TEST_IDS:
        ch = run_sim(tid, A_leak_cm2=A_leak_cm2, K_minor_mult=K_minor_mult)
        ss = extract_ss(ch, tid)
        ss_mdot_sim[tid] = ss['mdot_ss']
        ss_press_sim[tid] = ss['press_ss']

    J_mdot = 0.0
    J_press = 0.0
    details = {'per_test': {}}

    for tid in TEST_IDS:
        m_sim = ss_mdot_sim[tid]
        m_exp = exp_ss_mdot[tid]
        m_std = exp_ss_mdot_std[tid]
        if np.isnan(m_exp) or m_exp <= 0:
            continue
        if np.isnan(m_std) or m_std <= 0:
            m_std = 0.2 * m_exp
        r_m = ((m_sim - m_exp) / m_std) ** 2
        J_mdot += r_m

        ct = exp_data[tid]['chimney_type']
        r_p = 0.0
        n_p = 0
        for s in PRESS_SENSORS[ct]:
            if s in EXCLUDE_PRESS_SENSORS:
                continue
            ps = ss_press_sim[tid].get(s, np.nan)
            pe = exp_ss_press[tid].get(s, np.nan)
            if np.isnan(ps) or np.isnan(pe):
                continue
            r_p += ((ps - pe) / SIGMA_P) ** 2
            n_p += 1
        J_press += r_p

        details['per_test'][tid] = {
            'mdot_sim': m_sim, 'mdot_exp': m_exp, 'mdot_resid_sq': r_m,
            'press_resid_sq': r_p, 'n_press': n_p,
        }

    J_A = W_MDOT * J_mdot + W_PRESS * J_press
    details['J_mdot'] = J_mdot
    details['J_press'] = J_press
    details['J_A'] = J_A
    return J_A, details


# --- Cache ---
_cache = {}

def _hash(A_leak, K_mult):
    s = f'{A_leak:.4f}_{K_mult:.4f}'
    return hashlib.md5(s.encode()).hexdigest()[:12]

def eval_cached(A_leak_cm2, K_minor_mult):
    h = _hash(A_leak_cm2, K_minor_mult)
    if h in _cache:
        return _cache[h]
    J, details = compute_objective(A_leak_cm2, K_minor_mult)
    _cache[h] = (J, details)
    return J, details


print('Objective + cache defined.')

In [ ]:
print(f'=== Coarse sweep: {N_COARSE} LHS samples ===')
rng = np.random.default_rng(RNG_SEED)
samples = latin_hypercube_2d(N_COARSE, [A_LEAK_BOUNDS, K_MINOR_MULT_BOUNDS], rng)

sweep_results = []
t0 = _time.time()
for i, (al, km) in enumerate(samples):
    t1 = _time.time()
    J, details = eval_cached(al, km)
    elapsed = _time.time() - t1
    sweep_results.append({
        'A_leak_cm2': float(al), 'K_minor_mult': float(km),
        'J_A': float(J), 'J_mdot': details['J_mdot'], 'J_press': details['J_press'],
        'elapsed_s': round(elapsed, 1),
    })
    print(f'  [{i+1}/{N_COARSE}] J={J:.4f}  A_leak={al:.1f}  K_mult={km:.3f}  ({elapsed:.0f}s)')

print(f'Coarse sweep done in {_time.time()-t0:.0f}s')

# Sort and show top 10
sweep_sorted = sorted(sweep_results, key=lambda r: r['J_A'])
print('')
print('Top 10:')
for i, r in enumerate(sweep_sorted[:10]):
    print(f'  #{i+1}: J={r["J_A"]:.4f}  A_leak={r["A_leak_cm2"]:.1f}  '
          f'K_mult={r["K_minor_mult"]:.3f}  (J_m={r["J_mdot"]:.3f} J_p={r["J_press"]:.3f})')

# Save sweep
sweep_dir = OUT_ROOT / 'sweep'
sweep_dir.mkdir(parents=True, exist_ok=True)
with open(sweep_dir / 'coarse_results.json', 'w') as f:
    json.dump(sweep_results, f, indent=2)
print(f'Saved to {sweep_dir / "coarse_results.json"}')

In [ ]:
print(f'=== Local refinement: top {N_REFINE_TOP} candidates ===')

lo = np.array([A_LEAK_BOUNDS[0], K_MINOR_MULT_BOUNDS[0]])
hi = np.array([A_LEAK_BOUNDS[1], K_MINOR_MULT_BOUNDS[1]])

refined_results = []
for rank, candidate in enumerate(sweep_sorted[:N_REFINE_TOP]):
    x0 = np.array([candidate['A_leak_cm2'], candidate['K_minor_mult']])
    best_this = {'J': candidate['J_A'], 'x': x0.copy()}
    print('')
    print(f'--- Candidate #{rank+1}: J={candidate["J_A"]:.4f}  '
          f'A_leak={x0[0]:.1f}  K_mult={x0[1]:.3f} ---')

    def _obj(x):
        xc = np.clip(x, lo, hi)
        J, _ = eval_cached(float(xc[0]), float(xc[1]))
        if J < best_this['J']:
            best_this['J'] = J
            best_this['x'] = xc.copy()
            print(f'    refine: J={J:.4f}  A_leak={xc[0]:.1f}  K_mult={xc[1]:.3f}')
        return J

    res = minimize(_obj, x0, method='Nelder-Mead',
                   options={'maxiter': REFINE_MAXITER, 'xatol': 0.5, 'fatol': 1e-3,
                            'adaptive': True})
    xb = np.clip(res.x, lo, hi)
    Jb, det = eval_cached(float(xb[0]), float(xb[1]))
    refined_results.append({
        'A_leak_cm2': float(xb[0]), 'K_minor_mult': float(xb[1]),
        'J_A': float(Jb), 'details': det,
        'nfev': int(res.nfev), 'success': bool(res.success),
    })
    print(f'    Final: J={Jb:.4f}  A_leak={xb[0]:.1f}  K_mult={xb[1]:.3f}  '
          f'(nfev={res.nfev})')

# Select best
best = min(refined_results, key=lambda r: r['J_A'])
BEST_A_LEAK = best['A_leak_cm2']
BEST_K_MULT = best['K_minor_mult']
BEST_J_A    = best['J_A']

# Baseline J for comparison
J_baseline, _ = eval_cached(BASELINE_A_LEAK_CM2, BASELINE_K_MINOR_MULT)

print('')
print('=== BEST Stage A result ===')
print(f'  A_leak_cm2   = {BEST_A_LEAK:.2f}')
print(f'  K_minor_mult = {BEST_K_MULT:.4f}')
print(f'  J_A          = {BEST_J_A:.4f}')
print(f'  Baseline J_A = {J_baseline:.4f}')
improvement = (1 - BEST_J_A / max(J_baseline, 1e-12)) * 100
print(f'  Improvement  = {improvement:.1f}%')

# Bounds check
warnings = []
if abs(BEST_A_LEAK - A_LEAK_BOUNDS[0]) < 1.0:
    warnings.append(f'WARNING: A_leak_cm2={BEST_A_LEAK:.1f} is at LOWER bound!')
if abs(BEST_A_LEAK - A_LEAK_BOUNDS[1]) < 1.0:
    warnings.append(f'WARNING: A_leak_cm2={BEST_A_LEAK:.1f} is at UPPER bound!')
if abs(BEST_K_MULT - K_MINOR_MULT_BOUNDS[0]) < 0.02:
    warnings.append(f'WARNING: K_minor_mult={BEST_K_MULT:.3f} is at LOWER bound!')
if abs(BEST_K_MULT - K_MINOR_MULT_BOUNDS[1]) < 0.02:
    warnings.append(f'WARNING: K_minor_mult={BEST_K_MULT:.3f} is at UPPER bound!')
for w in warnings:
    print(f'  ** {w}')
if not warnings:
    print('  Parameters are away from bounds (good).')

# Save
with open(sweep_dir / 'best_params.json', 'w') as f:
    json.dump({'A_leak_cm2': BEST_A_LEAK, 'K_minor_mult': BEST_K_MULT,
               'J_A': BEST_J_A, 'J_baseline': J_baseline}, f, indent=2)

---
## Part 3: Diagnostic Visual Comparison

Side-by-side plots: **baseline** (dashed) vs **calibrated** (solid) vs **experiment** (black).

In [ ]:
# Run calibrated simulations
print(f'Running calibrated simulations: A_leak={BEST_A_LEAK:.1f}, K_mult={BEST_K_MULT:.3f}')
calibrated = {}
for tid in TEST_IDS:
    ch = run_sim(tid, A_leak_cm2=BEST_A_LEAK, K_minor_mult=BEST_K_MULT)
    ss = extract_ss(ch, tid)
    press = extract_pressure(ch, tid)
    calibrated[tid] = {'ch': ch, 'ss': ss, 'press': press}
    print(f'  {tid}: mdot_ss={ss["mdot_ss"]:.1f} g/s')
print('Done.')

compare_dir = OUT_ROOT / 'comparison'
compare_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ch_b = baseline[tid]['ch']
    ch_c = calibrated[tid]['ch']
    t_sim_b = np.arange(ch_b.T_gas.shape[0]) * ch_b.dt
    t_sim_c = np.arange(ch_c.T_gas.shape[0]) * ch_c.dt
    door_times = get_door_event_times(tid)
    sensors = TEMP_SENSORS[ct]

    fig, axes = plt.subplots(len(sensors), 1, figsize=(12, 3*len(sensors)), sharex=True)
    if len(sensors) == 1: axes = [axes]

    for ax, sensor in zip(axes, sensors):
        td = exp_data[tid]['temperature'][sensor]
        et = np.asarray(td['time_s'], dtype=float)
        ev = np.asarray(td['temperature_C'], dtype=float)
        seg_b = find_segment(ch_b, *SENSOR_MAP[ct][sensor])
        seg_c = find_segment(ch_c, *SENSOR_MAP[ct][sensor])

        ax.plot(et/60, ev, 'k-', lw=0.8, alpha=0.7, label='Exp')
        ax.plot(t_sim_b/60, ch_b.T_gas[:, seg_b]-273.15, color=SENSOR_COLORS[sensor],
                ls='--', lw=1.0, alpha=0.6, label='Baseline')
        ax.plot(t_sim_c/60, ch_c.T_gas[:, seg_c]-273.15, color=SENSOR_COLORS[sensor],
                ls='-', lw=1.3, label='Calibrated')
        for dt_ in door_times:
            ax.axvline(dt_/60, color='gray', ls=':', alpha=0.4)
        ax.set_ylabel(f'{sensor} T (C)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (min)')
    axes[0].set_title(f'Temperature: {tid} ({ct}) -- Baseline vs Calibrated')
    savefig(fig, compare_dir / f'{tid}_T_compare.png')
    plt.show()

In [ ]:
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    press_b = baseline[tid]['press']
    press_c = calibrated[tid]['press']
    door_times = get_door_event_times(tid)
    sensors = PRESS_SENSORS[ct]

    fig, axes = plt.subplots(len(sensors), 1, figsize=(12, 2.8*len(sensors)), sharex=True)
    if len(sensors) == 1: axes = [axes]

    for ax, sensor in zip(axes, sensors):
        pd_ = exp_data[tid]['pressure'][sensor]
        et = np.asarray(pd_['time_s'], dtype=float)
        ev = np.asarray(pd_['pressure_Pa'], dtype=float)
        pt_b, pv_b = press_b[sensor]
        pt_c, pv_c = press_c[sensor]

        ax.plot(et/60, ev, 'k-', lw=0.8, alpha=0.7, label='Exp')
        ax.plot(pt_b/60, pv_b, color=SENSOR_COLORS[sensor], ls='--', lw=1.0,
                alpha=0.6, label='Baseline')
        ax.plot(pt_c/60, pv_c, color=SENSOR_COLORS[sensor], ls='-', lw=1.3,
                label='Calibrated')
        ax.axhline(0, color='gray', ls=':', alpha=0.4)
        for dt_ in door_times:
            ax.axvline(dt_/60, color='gray', ls=':', alpha=0.4)
        ax.set_ylabel(f'{sensor} P (Pa)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (min)')
    axes[0].set_title(f'Pressure: {tid} ({ct}) -- Baseline vs Calibrated')
    savefig(fig, compare_dir / f'{tid}_P_compare.png')
    plt.show()

In [ ]:
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ch_b = baseline[tid]['ch']
    ch_c = calibrated[tid]['ch']
    ss_b = baseline[tid]['ss']
    ss_c = calibrated[tid]['ss']

    sensors_p = PRESS_SENSORS[ct]
    sensors_t = TEMP_SENSORS[ct]

    heights_p = [get_sensor_elevation_m(ch_b, tid, s) for s in sensors_p]
    heights_t = [get_sensor_elevation_m(ch_b, tid, s) for s in sensors_t]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Pressure profile
    vals_p_exp = [exp_ss_press[tid].get(s, np.nan) for s in sensors_p]
    vals_p_bas = [ss_b['press_ss'][s] for s in sensors_p]
    vals_p_cal = [ss_c['press_ss'][s] for s in sensors_p]
    ax1.plot(vals_p_exp, heights_p, 'ko-', lw=1.2, ms=6, label='Exp')
    ax1.plot(vals_p_bas, heights_p, 's--', color='C0', lw=1.0, ms=5, alpha=0.6, label='Baseline')
    ax1.plot(vals_p_cal, heights_p, 'D-', color='C2', lw=1.3, ms=5, label='Calibrated')
    ax1.set_xlabel('Gauge Pressure (Pa)')
    ax1.set_ylabel('Height (m)')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f'{tid}: SS Pressure Profile')

    # Temperature profile
    vals_t_exp = [exp_ss_temp[tid].get(s, np.nan) for s in sensors_t]
    vals_t_bas = [ss_b['temp_ss'][s] for s in sensors_t]
    vals_t_cal = [ss_c['temp_ss'][s] for s in sensors_t]
    ax2.plot(vals_t_exp, heights_t, 'ko-', lw=1.2, ms=6, label='Exp')
    ax2.plot(vals_t_bas, heights_t, 's--', color='C0', lw=1.0, ms=5, alpha=0.6, label='Baseline')
    ax2.plot(vals_t_cal, heights_t, 'D-', color='C2', lw=1.3, ms=5, label='Calibrated')
    ax2.set_xlabel('Temperature (C)')
    ax2.set_ylabel('Height (m)')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_title(f'{tid}: SS Temperature Profile')

    savefig(fig, compare_dir / f'{tid}_ss_profiles_compare.png')
    plt.show()

---
## Part 4: Physics Sanity Checks

mdot ratios, pressure bias, parameter sensitivity, bounds warnings.

In [ ]:
print('=== Physics Sanity Checks ===')
print('')

# mdot ratios
print('SS mdot ratios (sim/exp):')
for tid in TEST_IDS:
    m_s = calibrated[tid]['ss']['mdot_ss']
    m_e = exp_ss_mdot[tid]
    ratio = m_s / m_e if m_e > 0 else np.nan
    print(f'  {tid}: sim={m_s:.1f}  exp={m_e:.1f}  ratio={ratio:.3f}')

# Mean pressure bias
print('')
print('Mean pressure bias (sim-exp, closed-door, excl TC1):')
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    biases = []
    for s in PRESS_SENSORS[ct]:
        if s in EXCLUDE_PRESS_SENSORS:
            continue
        ps = calibrated[tid]['ss']['press_ss'].get(s, np.nan)
        pe = exp_ss_press[tid].get(s, np.nan)
        if not (np.isnan(ps) or np.isnan(pe)):
            biases.append(ps - pe)
    mean_bias = np.mean(biases) if biases else np.nan
    print(f'  {tid}: mean bias = {mean_bias:+.2f} Pa  (n={len(biases)} sensors)')

# Parameter summary
print('')
print('Calibrated parameters:')
print(f'  A_leak_cm2   = {BEST_A_LEAK:.2f}  (bounds: {A_LEAK_BOUNDS})')
print(f'  K_minor_mult = {BEST_K_MULT:.4f}  (bounds: {K_MINOR_MULT_BOUNDS})')
print(f'  Cd           = {CD_FIXED} (FIXED)')
print(f'  K_stove      = {K_STOVE_FIXED} (FIXED)')

# Bounds warnings
for w in warnings:
    print(f'  ** {w}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

als = [r['A_leak_cm2'] for r in sweep_results]
kms = [r['K_minor_mult'] for r in sweep_results]
Js  = [r['J_A'] for r in sweep_results]

# J vs A_leak
sc1 = ax1.scatter(als, Js, c=kms, cmap='viridis', s=40, alpha=0.8, edgecolors='k', lw=0.3)
ax1.scatter([BEST_A_LEAK], [BEST_J_A], s=120, color='red', marker='*', zorder=5, label='Best')
ax1.set_xlabel('A_leak_cm2')
ax1.set_ylabel('J_A')
ax1.set_title('Sensitivity: J_A vs A_leak')
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.colorbar(sc1, ax=ax1, label='K_minor_mult')

# J vs K_minor_mult
sc2 = ax2.scatter(kms, Js, c=als, cmap='plasma', s=40, alpha=0.8, edgecolors='k', lw=0.3)
ax2.scatter([BEST_K_MULT], [BEST_J_A], s=120, color='red', marker='*', zorder=5, label='Best')
ax2.set_xlabel('K_minor_mult')
ax2.set_ylabel('J_A')
ax2.set_title('Sensitivity: J_A vs K_minor_mult')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.colorbar(sc2, ax=ax2, label='A_leak_cm2')

savefig(fig, OUT_ROOT / 'sensitivity_scatter.png')
plt.show()

In [ ]:
# 2D objective landscape from sweep data
from scipy.interpolate import griddata

als_arr = np.array([r['A_leak_cm2'] for r in sweep_results])
kms_arr = np.array([r['K_minor_mult'] for r in sweep_results])
Js_arr  = np.array([r['J_A'] for r in sweep_results])

if len(sweep_results) >= 10:
    grid_al = np.linspace(A_LEAK_BOUNDS[0], A_LEAK_BOUNDS[1], 50)
    grid_km = np.linspace(K_MINOR_MULT_BOUNDS[0], K_MINOR_MULT_BOUNDS[1], 50)
    AL, KM = np.meshgrid(grid_al, grid_km)
    J_grid = griddata((als_arr, kms_arr), np.log10(np.maximum(Js_arr, 1e-6)),
                       (AL, KM), method='linear')

    fig, ax = plt.subplots(figsize=(8, 6))
    cf = ax.contourf(AL, KM, J_grid, levels=20, cmap='RdYlGn_r', alpha=0.8)
    ax.scatter(als_arr, kms_arr, c='k', s=15, alpha=0.5, label='LHS samples')
    ax.scatter([BEST_A_LEAK], [BEST_K_MULT], s=150, color='red', marker='*',
               zorder=5, label='Best')
    ax.scatter([BASELINE_A_LEAK_CM2], [BASELINE_K_MINOR_MULT], s=100,
               color='blue', marker='o', zorder=5, label='Baseline')
    ax.set_xlabel('A_leak_cm2')
    ax.set_ylabel('K_minor_mult')
    ax.set_title('Objective landscape: log10(J_A)')
    ax.legend(loc='upper right')
    plt.colorbar(cf, ax=ax, label='log10(J_A)')
    savefig(fig, OUT_ROOT / 'objective_landscape.png')
    plt.show()
else:
    print('Not enough samples for contour plot (need >= 10).')

---
## Report

In [ ]:
# Compute final metrics
mdot_ratios = {}
press_biases = {}
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    m_s = calibrated[tid]['ss']['mdot_ss']
    m_e = exp_ss_mdot[tid]
    mdot_ratios[tid] = round(m_s / m_e, 3) if m_e > 0 else np.nan
    biases = []
    for s in PRESS_SENSORS[ct]:
        if s in EXCLUDE_PRESS_SENSORS:
            continue
        ps = calibrated[tid]['ss']['press_ss'].get(s, np.nan)
        pe = exp_ss_press[tid].get(s, np.nan)
        if not (np.isnan(ps) or np.isnan(pe)):
            biases.append(ps - pe)
    press_biases[tid] = round(np.mean(biases), 2) if biases else np.nan

# Determine which knob mattered more
# Compare J sensitivity: run best with each knob at baseline
J_at_best = BEST_J_A
J_leak_only, _ = eval_cached(BEST_A_LEAK, BASELINE_K_MINOR_MULT)
J_kmult_only, _ = eval_cached(BASELINE_A_LEAK_CM2, BEST_K_MULT)

sensitivity_leak  = abs(J_baseline - J_leak_only)
sensitivity_kmult = abs(J_baseline - J_kmult_only)
dominant_knob = 'A_leak_cm2' if sensitivity_leak > sensitivity_kmult else 'K_minor_mult'

# Degeneracy check: correlation in sweep results
if len(sweep_results) > 5:
    top5 = sweep_sorted[:5]
    al_top = [r['A_leak_cm2'] for r in top5]
    km_top = [r['K_minor_mult'] for r in top5]
    corr = np.corrcoef(al_top, km_top)[0, 1] if len(al_top) > 2 else 0
    degeneracy_note = (f'Top-5 A_leak/K_mult correlation: {corr:.2f}. '
                       + ('Strong degeneracy detected.' if abs(corr) > 0.7
                          else 'Low degeneracy.'))
else:
    degeneracy_note = 'Insufficient samples for degeneracy check.'

# Pressure shape
press_improved = sum(
    1 for tid in TEST_IDS
    if abs(press_biases.get(tid, 99)) < abs(
        np.mean([baseline[tid]['ss']['press_ss'].get(s, 0) - exp_ss_press[tid].get(s, 0)
                 for s in PRESS_SENSORS[exp_data[tid]['chimney_type']]
                 if s not in EXCLUDE_PRESS_SENSORS]) if True else 99
    )
)

report = f"""
-----BEGIN REPORT-----
Best Stage A Parameters:
  A_leak_cm2   = {BEST_A_LEAK:.2f}
  K_minor_mult = {BEST_K_MULT:.4f}
  Cd           = {CD_FIXED} (fixed)
  K_stove      = {K_STOVE_FIXED} (fixed)

Baseline J_A = {J_baseline:.4f}
Best J_A     = {BEST_J_A:.4f}
Improvement  = {(1 - BEST_J_A / max(J_baseline, 1e-12)) * 100:.1f}%

mdot ratios per test (sim/exp):
  {json.dumps(mdot_ratios, indent=4)}

Mean pressure bias per test (Pa, excl TC1):
  {json.dumps(press_biases, indent=4)}

Interpretation:
  - Dominant knob: {dominant_knob}
    (dJ from A_leak alone: {sensitivity_leak:.2f}, from K_mult alone: {sensitivity_kmult:.2f})
  - {degeneracy_note}
  - Pressure shape improved for {press_improved}/{len(TEST_IDS)} tests.

Sweep: {N_COARSE} coarse + {N_REFINE_TOP} x {REFINE_MAXITER} refinement iterations
Seed: {RNG_SEED}
Output: {OUT_ROOT}
-----END REPORT-----
"""
print(report)

# Save report
with open(OUT_ROOT / 'report.txt', 'w') as f:
    f.write(report)
print(f'Report saved to {OUT_ROOT / "report.txt"}')